# Run GenX PCM Cases

In [ ]:
from ipywidgets import Dropdown, SelectMultiple
from upath import UPath
from src import runner
from ipyfilechooser import FileChooser
import xlwings as xw
from loguru import logger
from tqdm.notebook import trange, tqdm

In [ ]:
genx_wb = FileChooser(default_path=".", default_filename="Kentucky Load Resource Model.xlsb", title="Connect to a GenX spreadsheet: ", filter_pattern="*.xls*", show_hidden=False)
genx_wb

In [ ]:
selected_cases = SelectMultiple(
    options=runner.get_solved_cases(UPath("./cases")),
    description="Available CEM cases: ",
    rows=10,
    layout=dict(width="max-content"),
    style=dict(description_width="max-content"),
)
selected_cases

In [ ]:
pcm_year = Dropdown(options=range(2025, 2051), value=2035, description="PCM year: ", style=dict(description_width="max-content"),)
pcm_year

In [ ]:
logger.info(f"Opening {genx_wb.value} in new Excel instance")

folders = []

if UPath(genx_wb.value).parts[-1] in xw.books:
    xw.books[UPath(genx_wb.value).parts[-1]].save()

with xw.App() as xw_sandbox:
    pcm_wb = xw.apps[xw_sandbox.pid].books.open(genx_wb.value)

    for cem_case in tqdm(selected_cases.value, desc="Saving PCM cases"):
        cem_path = UPath(cem_case)
        cem_case = cem_path.stem

        # Load CEM portfolio results
        runner.load_case_results(
            wb=pcm_wb,
            base_folder=cem_path,
            save_view=False,
        )

        # Activate the corresponding case configurations (identified solely by the case's name, so unique case names matter)
        pcm_wb.sheets["GenX Settings"].range("ActiveYear").value = pcm_year.value
        pcm_wb.sheets["GenX Settings"].range("CaseName").value = cem_case
        pcm_wb.sheets["PCM Settings"].range("PCMPortfolio").value = str(cem_path.absolute())

        # Turn off all other modeled years except the PCM year
        pcm_wb.sheets["GenX Settings"].range("ModeledYears[Modeled]").value = False

        row_index = pcm_wb.sheets["GenX Settings"].range("ModeledYears[Planning Period]").value.index(pcm_year.value)
        pcm_wb.sheets["GenX Settings"].range("ModeledYears[Modeled]").offset(row_index, 0).resize(1, 1).value = True

        # Save PCM case settings
        folders.append(runner.save_multistage_case(wb=pcm_wb))
    
    pcm_wb.close()

# Run PCM cases in parallel
results = runner.run_cases_with_streaming_logs(folders)